# 06.6–06.7 Identity and Membership Operators

Four operators, two jobs:

- **`is` / `is not`** — are these the *same object*?
- **`in` / `not in`** — is this value *inside* that container?

Both are simple. Both have edge cases worth knowing.

**35 numbered examples.**

## Theory

### `is` compares identity

```python
a is b      # exactly equivalent to  id(a) == id(b)
```

There is no magic and **no way to override it**. That guarantee is exactly why
`x is None` is the correct idiom — a class can lie about `==`, but not about
identity.

**Use `is` only for:** `None`, `True`, `False`, sentinels, and genuine
"same object" checks. Never for comparing values.

### `in` uses a chain of fallbacks

`x in container` tries, in order:

1. `container.__contains__(x)` if defined
2. Otherwise iterate with `__iter__` and compare each item
3. Otherwise use `__getitem__` with increasing indices

Which means `in` is **O(1)** for sets and dicts, but **O(n)** for lists and
tuples. That difference matters enormously at scale.

### The identity-before-equality detail

When `in` compares items, it checks `x is item or x == item`. The identity check
comes first — which is why `nan in [nan]` is `True` even though `nan == nan` is
`False`.

### What `in` searches

- **Sequences** — the elements
- **Dicts** — the **keys**, not the values
- **Strings** — **substrings**, not just characters

In [ ]:
# EXAMPLE 1-6: is vs ==.
print("EXAMPLE 1-6: identity vs equality")
print("")

list_a = [1, 2, 3]
list_b = [1, 2, 3]
list_c = list_a

print("   1. list_a == list_b ->", list_a == list_b, "(same value)")
print("   2. list_a is list_b ->", list_a is list_b, "(different objects)")
print("   3. list_a is list_c ->", list_a is list_c, "(same object)")
print("")
print("   4. id(list_a):", id(list_a))
print("      id(list_b):", id(list_b))
print("      id(list_c):", id(list_c))
print("")
print("   5. `is` is literally an id comparison:",
      (list_a is list_c) == (id(list_a) == id(list_c)))

# 6. is not.
print("   6. list_a is not list_b ->", list_a is not list_b)

In [ ]:
# EXAMPLE 7-12: why `is` for None.
print("EXAMPLE 7-12: testing for None")
print("")

value = None
print("   7. value is None ->", value is None, "<- correct")
print("   8. value == None ->", value == None, "<- works, but E711")


class Deceptive:
    """Claims equality with everything."""

    def __eq__(self, other):
        return True


tricky = Deceptive()
print("")
print("   9.  a class can lie about ==:")
print("       tricky == None ->", tricky == None, "<- wrong answer")
print("   10. but not about identity:")
print("       tricky is None ->", tricky is None)

# 11-12. The singletons.
print("")
print("   11. the three singletons always work with `is`:")
print("       None is None   ->", None is None)
print("       True is True   ->", True is True)
print("       False is False ->", False is False)

print("")
print("   12. custom sentinels rely on the same guarantee:")
MISSING = object()
setting = MISSING
print("       setting is MISSING ->", setting is MISSING)

In [ ]:
# EXAMPLE 13-18: where `is` gives surprising answers.
print("EXAMPLE 13-18: interning surprises")
print("")

# 13-14. Small integers are cached.
small_a = int("100")
small_b = int("100")
large_a = int("1000")
large_b = int("1000")

print("   13. int('100')  is int('100')  ->", small_a is small_b)
print("   14. int('1000') is int('1000') ->", large_a is large_b)
print("       CPython caches -5 to 256 only.")

# 15-16. String interning.
text_a = "hello"
text_b = "hello"
built = "".join(["h", "e", "l", "l", "o"])

print("")
print("   15. 'hello' is 'hello'        ->", text_a is text_b)
print("   16. built at runtime          ->", built is text_a)
print("       but equal by value        ->", built == text_a)

# 17. Empty immutables are often shared.
print("")
print("   17. empty immutables:")
print("       () is ()     ->", tuple() is tuple())
print("       [] is []     ->", [] is [])

# 18. The rule.
print("")
print("   18. NEVER use `is` to compare values. Python warns you:")
print("       writing `x is 100` raises SyntaxWarning in 3.8+")

## Membership: `in` and `not in`

In [ ]:
# EXAMPLE 19-24: in across container types.
print("EXAMPLE 19-24: what `in` searches")
print("")

# 19. Lists and tuples search elements.
print("   19. list:   3 in [1, 2, 3]      ->", 3 in [1, 2, 3])
print("   20. tuple:  3 in (1, 2, 3)      ->", 3 in (1, 2, 3))

# 21. Sets.
print("   21. set:    3 in {1, 2, 3}      ->", 3 in {1, 2, 3})

# 22. Dicts search KEYS, not values.
prices = {"apple": 10, "banana": 5}
print("")
print("   22. dict searches KEYS:")
print("       'apple' in prices ->", "apple" in prices)
print("       10 in prices      ->", 10 in prices, "<- 10 is a VALUE")
print("       10 in prices.values() ->", 10 in prices.values())

# 23. Strings search SUBSTRINGS.
print("")
print("   23. string searches substrings, not just characters:")
print("       'ell' in 'hello'  ->", "ell" in "hello")
print("       'lo' in 'hello'   ->", "lo" in "hello")
print("       'ol' in 'hello'   ->", "ol" in "hello", "<- order matters")

# 24. not in.
print("")
print("   24. not in is a single operator:")
print("       4 not in [1, 2, 3]   ->", 4 not in [1, 2, 3])
print("       not (4 in [1, 2, 3]) ->", not (4 in [1, 2, 3]), "<- same, worse")

In [ ]:
import time

# EXAMPLE 25-28: the performance difference.
print("EXAMPLE 25-28: in is O(1) for sets, O(n) for lists")
print("")

size = 100000
as_list = list(range(size))
as_set = set(as_list)
as_dict = {value: None for value in as_list}

# Search for the worst-case item - the last one.
target = size - 1
repetitions = 200


def time_lookup(container):
    """Time repeated membership tests."""
    start = time.perf_counter()
    for _ in range(repetitions):
        target in container
    return (time.perf_counter() - start) * 1000


list_time = time_lookup(as_list)
set_time = time_lookup(as_set)
dict_time = time_lookup(as_dict)

print(f"   searching {size:,} items, {repetitions} times:")
print("")
print(f"   25. list: {list_time:>8.2f} ms   O(n) - checks every element")
print(f"   26. set:  {set_time:>8.2f} ms   O(1) - hashes straight to it")
print(f"   27. dict: {dict_time:>8.2f} ms   O(1) - same hash table")

if set_time > 0:
    print("")
    print(f"   28. the set was about {list_time / set_time:,.0f}x faster")
print("       If you test membership repeatedly, use a set.")

In [ ]:
# EXAMPLE 29-35: edge cases.
print("EXAMPLE 29-35: edge cases")
print("")

# 29. in uses identity BEFORE equality.
nan = float("nan")
print("   29. nan == nan      ->", nan == nan)
print("       nan in [nan]    ->", nan in [nan], "<- identity matched first")
print("       nan in [float('nan')] ->", nan in [float("nan")])

# 30. True and 1 are interchangeable.
print("")
print("   30. True in [1]  ->", True in [1], "<- bool is an int")
print("       1 in [True]  ->", 1 in [True])

# 31. Nested containers match whole elements.
nested = [[1, 2], [3, 4]]
print("")
print("   31. [1, 2] in [[1,2],[3,4]] ->", [1, 2] in nested)
print("       1 in [[1,2],[3,4]]      ->", 1 in nested, "<- not flattened")

# 32. Empty containers.
print("")
print("   32. anything in an empty container ->", 1 in [])
print("       '' in 'hello'                  ->", "" in "hello", "<- always True")

# 33. Custom __contains__.
class Range:
    """A range that defines its own membership test."""

    def __init__(self, low, high):
        self.low = low
        self.high = high

    def __contains__(self, value):
        # Called directly by `in`.
        return self.low <= value <= self.high


span = Range(1, 10)
print("")
print("   33. custom __contains__:")
print("       5 in Range(1, 10)  ->", 5 in span)
print("       15 in Range(1, 10) ->", 15 in span)

# 34. Without __contains__, Python falls back to iteration.
class Countdown:
    """Iterable but with no __contains__."""

    def __iter__(self):
        return iter([3, 2, 1])


print("")
print("   34. falls back to iteration:")
print("       2 in Countdown() ->", 2 in Countdown())

# 35. in works on generators - but CONSUMES them.
generator = (value for value in range(5))
print("")
print("   35. in consumes a generator:")
print("       3 in generator ->", 3 in generator)
print("       what remains:", list(generator), "<- 0,1,2,3 were consumed")

## Takeaways

1. `is` compares **identity** and is exactly `id(a) == id(b)`. It cannot be
   overridden.
2. Use `is` only for `None`, `True`, `False`, sentinels and genuine same-object
   checks — **never for values**.
3. CPython **caches** small integers and interns some strings, so `is` can be
   `True` for separate literals. Never rely on it.
4. `in` searches **keys** for dicts and **substrings** for strings.
5. `in` is **O(1)** for sets and dicts, **O(n)** for lists and tuples — often
   thousands of times faster at scale.
6. `in` checks **identity before equality**, which is why `nan in [nan]` is
   `True`.
7. Classes control `in` via `__contains__`, falling back to iteration.
8. `in` on a generator **consumes** it up to the match.

## Try it yourself

1. Write a class whose `__eq__` always returns `True`. Confirm `is` still works.
2. Time `in` on a 100,000-item list versus a set.
3. Explain why `nan in [nan]` is `True` but `nan == nan` is `False`.
4. Write a class with `__contains__` for even numbers.
5. Test `3 in gen` on a generator, then print what remains.